In [39]:
import cv2
import mediapipe as mp
import pyautogui
import random
import util
import screen_brightness_control as sbc
import time
from pynput.mouse import Button, Controller
mouse = Controller()

screen_width, screen_height = pyautogui.size()

print("Screen Width:", screen_width)
print("Screen Height:", screen_height)

Screen Width: 1920
Screen Height: 1080


In [40]:
current_brightness = sbc.get_brightness()[0]

last_brightness_update = 0

In [41]:
def is_brightness_up(landmark_list):
    
    return (
        landmark_list[4][1] < landmark_list[3][1] and
        landmark_list[8][1] > landmark_list[6][1] and
        landmark_list[12][1] > landmark_list[10][1] and
        landmark_list[16][1] > landmark_list[14][1] and
        landmark_list[20][1] > landmark_list[18][1]
    )

In [43]:
def is_brightness_down(landmark_list):
    
    return (
        landmark_list[4][1] > landmark_list[3][1] and
        landmark_list[8][1] > landmark_list[6][1] and
        landmark_list[12][1] > landmark_list[10][1] and
        landmark_list[16][1] > landmark_list[14][1] and
        landmark_list[20][1] > landmark_list[18][1]
    )

In [46]:
def control_brightness(landmark_list, frame):
    
    global current_brightness, last_brightness_update

    current_time = time.time()

    if current_time - last_brightness_update < 0.5:
        return

    if is_brightness_up(landmark_list):

        current_brightness = min(100, current_brightness + 10)

    elif is_brightness_down(landmark_list):

        current_brightness = max(10, current_brightness - 10)

    else:
        return

    sbc.set_brightness(current_brightness)

    last_brightness_update = current_time

    cv2.putText(
        frame,
        f"Brightness: {current_brightness}%",
        (50, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

In [23]:
screen_width, screen_height = pyautogui.size()

In [47]:
mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
    max_num_hands=1
)

In [48]:
def find_finger_tip(processed):
    if processed.multi_hand_landmarks:
        hand_landmarks = processed.multi_hand_landmarks[0]  # Assuming only one hand is detected
        index_finger_tip = hand_landmarks.landmark[mpHands.HandLandmark.INDEX_FINGER_TIP]
        return index_finger_tip
    return None, None

In [49]:
def move_mouse(index_finger_tip):
    if index_finger_tip is not None:
        x = int(index_finger_tip.x * screen_width)
        y = int(index_finger_tip.y / 2 * screen_height)
        pyautogui.moveTo(x, y)

In [50]:
def is_left_click(landmark_list, thumb_index_dist):
    return (
            util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) < 50 and
            util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) > 90 and
            thumb_index_dist > 50
    )

In [51]:
def is_right_click(landmark_list, thumb_index_dist):
    return (
            util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) < 50 and
            util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) > 90  and
            thumb_index_dist > 50
    )

In [52]:
def is_double_click(landmark_list, thumb_index_dist):
    return (
            util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) < 50 and
            util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) < 50 and
            thumb_index_dist > 50
    )

In [53]:
def is_screenshot(landmark_list, thumb_index_dist):
    return (
            util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) < 50 and
            util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) < 50 and
            thumb_index_dist < 50
    )

In [54]:
def detect_gesture(frame, landmark_list, processed):
    if len(landmark_list) >= 21:

        index_finger_tip = find_finger_tip(processed)
        thumb_index_dist = util.get_distance([landmark_list[4], landmark_list[5]])

         # NEW: Scroll control
        if is_scroll_gesture(landmark_list):
            control_scroll(landmark_list, frame)


        elif is_brightness_up(landmark_list) or is_brightness_down(landmark_list):
    
            control_brightness(landmark_list, frame)

        elif util.get_distance([landmark_list[4], landmark_list[5]]) < 50 and util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) > 90:

            move_mouse(index_finger_tip)

        if util.get_distance([landmark_list[4], landmark_list[5]]) < 50  and util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) > 90:
            move_mouse(index_finger_tip)
        elif is_left_click(landmark_list,  thumb_index_dist):
            mouse.press(Button.left)
            mouse.release(Button.left)
            cv2.putText(frame, "Left Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        elif is_right_click(landmark_list, thumb_index_dist):
            mouse.press(Button.right)
            mouse.release(Button.right)
            cv2.putText(frame, "Right Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        elif is_double_click(landmark_list, thumb_index_dist):
            pyautogui.doubleClick()
            cv2.putText(frame, "Double Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
        elif is_screenshot(landmark_list,thumb_index_dist ):
            im1 = pyautogui.screenshot()
            label = random.randint(1, 1000)
            im1.save(f'my_screenshot_{label}.png')
            cv2.putText(frame, "Screenshot Taken", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)


In [55]:
def main():
    draw = mp.solutions.drawing_utils
    cap = cv2.VideoCapture(0)

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.flip(frame, 1)
            frameRGB = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            processed = hands.process(frameRGB)

            landmark_list = []
            if processed.multi_hand_landmarks:
                hand_landmarks = processed.multi_hand_landmarks[0]  # Assuming only one hand is detected
                draw.draw_landmarks(frame, hand_landmarks, mpHands.HAND_CONNECTIONS)
                for lm in hand_landmarks.landmark:
                    landmark_list.append((lm.x, lm.y))

            detect_gesture(frame, landmark_list, processed)

            cv2.imshow('Frame', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [56]:
def is_scroll_gesture(landmark_list):
    
    return (
        util.get_angle(landmark_list[5], landmark_list[6], landmark_list[8]) > 90 and
        util.get_angle(landmark_list[9], landmark_list[10], landmark_list[12]) > 90 and
        util.get_angle(landmark_list[13], landmark_list[14], landmark_list[16]) < 50 and
        util.get_angle(landmark_list[17], landmark_list[18], landmark_list[20]) < 50
    )

In [57]:
scroll_start_y = None
scroll_direction = 0

def control_scroll(landmark_list, frame):

    global scroll_start_y, scroll_direction

    current_y = landmark_list[9][1]

    # Set the starting hand position
    if scroll_start_y is None:
        scroll_start_y = current_y
        return

    movement = scroll_start_y - current_y

    if movement > 0.04:

        pyautogui.scroll(5)
        scroll_direction = 1

    elif movement < -0.04:

        pyautogui.scroll(-5)
        scroll_direction = -1

    else:
        scroll_direction = 0

    cv2.putText(
        frame,
        "Scrolling Up" if scroll_direction == 1 else
        "Scrolling Down" if scroll_direction == -1 else
        "Scroll Ready",
        (50, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 0),
        2
    )

In [58]:
if __name__ == '__main__':
    main()

KeyboardInterrupt: 